# 01 — Diagnóstico do schema DataJud

**Secção da dissertação:** 3.5.1 (amostra e cobertura) — etapa prévia obrigatória.

Este notebook **não assume** a estrutura do `.jsonl`. Lê as primeiras linhas,
lista caminhos de chaves aninhadas, classifica se cada linha é um processo ou
uma página Elasticsearch, e mede a presença dos campos esperados em *todo* o
ficheiro. Grava `data/processado/processos.parquet` para os notebooks seguintes.


In [1]:
# Arranque reproduzível: funciona a partir da raiz ou de notebooks/.
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def raiz_projeto() -> Path:
    cwd = Path.cwd().resolve()
    for cand in (cwd, cwd.parent, Path(".").resolve()):
        if (cand / "src" / "config.py").exists():
            return cand
    raise RuntimeError("Não encontrei a raiz do projecto (pasta src/).")

RAIZ = raiz_projeto()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.caminhos import garantir_directorias
from src.config import aviso_parametros_pendentes, parametros_efectivos
from src.graficos_apa import CatalogoAPA, aplicar_estilo_apa, formatar_p

DIRS = garantir_directorias(RAIZ)
PARAM = parametros_efectivos(DIRS["processado"])
aplicar_estilo_apa()

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 88)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)

print(aviso_parametros_pendentes(PARAM))
print("Raiz:", RAIZ)


AVISO METODOLÓGICO — parâmetros ainda não validados com o orientador.
  Limiar mínimo de movimentos: 5 (fallback provisório (config.py))
  Ano-base da normalização: None (ainda indefinido — será a primeira coorte observada)
  Coortes: (derivar dos dados) (derivar dos dados em tempo de execução)
  O índice produzido nesta corrida é operacional, não a versão final da dissertação.
Raiz: /workspace


In [2]:
from IPython.display import display, Markdown, HTML
import json
from collections import Counter

from src.io_datajud import (
    CAMPOS_ESPERADOS,
    caminhos_chaves,
    carregar_processos,
    diagnostico_cobertura,
    localizar_jsonl,
)

catalogo = CatalogoAPA(
    DIRS["figuras"], DIRS["tabelas"],
    DIRS["saidas"] / "relatorio_resultados.md",
    reiniciar_relatorio=True,
)


## Localização do ficheiro e inspeção das primeiras linhas

A extração de referência (`DATAJUD_API_v_final.ipynb`) grava **um processo por
linha** (já o `_source` do Elasticsearch, não a página `hits.hits`). Confirmamos
isto empiricamente — se o ficheiro for uma dump cru da API, o parser adapta-se.


In [3]:
caminho_jsonl = localizar_jsonl(DIRS["dados"])
print("Ficheiro:", caminho_jsonl)
print(f"Tamanho: {caminho_jsonl.stat().st_size / 1e6:.2f} MB")

N_AMOSTRA = 20
amostra_bruta = []
with caminho_jsonl.open(encoding="utf-8") as ficheiro:
    for i, linha in enumerate(ficheiro):
        if i >= N_AMOSTRA:
            break
        if linha.strip():
            amostra_bruta.append(json.loads(linha))

print(f"Linhas de amostra lidas: {len(amostra_bruta)}")


Ficheiro: /workspace/data/processos_trf2_baixa_definitiva_2025.jsonl
Tamanho: 20.22 MB
Linhas de amostra lidas: 20


Para cada linha: chaves de topo, caminhos aninhados (`movimentos[].codigo`, etc.)
e um `json.dumps` *truncado* (os movimentos completos tornariam a célula ilegível).


In [4]:
def truncar_processo(obj, n_mov=2):
    copia = dict(obj)
    if isinstance(copia.get("movimentos"), list):
        copia["movimentos"] = copia["movimentos"][:n_mov] + (
            [{"_truncado": f"... {len(obj['movimentos']) - n_mov} movimentos omitidos"}]
            if len(obj["movimentos"]) > n_mov else []
        )
    return copia

linhas_resumo = []
for i, obj in enumerate(amostra_bruta, start=1):
    chaves = sorted(obj.keys()) if isinstance(obj, dict) else []
    caminhos = sorted(set(caminhos_chaves(obj)))
    linhas_resumo.append({
        "linha": i,
        "n_chaves_topo": len(chaves),
        "chaves_topo": ", ".join(chaves),
        "tem_hits": "hits" in chaves,
        "tem_movimentos": "movimentos" in chaves,
        "tem_numeroProcesso": "numeroProcesso" in chaves,
        "n_caminhos": len(caminhos),
    })
    print("=" * 78)
    print(f"LINHA {i} — chaves de topo: {chaves}")
    print("Caminhos aninhados:")
    for c in caminhos:
        print("  ", c)
    if i <= 2:
        print("JSON truncado:")
        print(json.dumps(truncar_processo(obj), ensure_ascii=False, indent=2)[:2500])

quadro_amostra = pd.DataFrame(linhas_resumo)
display(quadro_amostra)


LINHA 1 — chaves de topo: ['@timestamp', 'assuntos', 'classe', 'dataAjuizamento', 'dataHoraUltimaAtualizacao', 'formato', 'grau', 'id', 'movimentos', 'nivelSigilo', 'numeroProcesso', 'orgaoJulgador', 'sistema', 'tribunal']
Caminhos aninhados:
   @timestamp
   assuntos
   assuntos[]
   assuntos[].codigo
   assuntos[].nome
   classe
   classe.codigo
   classe.nome
   dataAjuizamento
   dataHoraUltimaAtualizacao
   formato
   formato.codigo
   formato.nome
   grau
   id
   movimentos
   movimentos[]
   movimentos[].codigo
   movimentos[].complementosTabelados
   movimentos[].complementosTabelados[]
   movimentos[].complementosTabelados[].codigo
   movimentos[].complementosTabelados[].descricao
   movimentos[].complementosTabelados[].nome
   movimentos[].complementosTabelados[].valor
   movimentos[].dataHora
   movimentos[].nome
   movimentos[].orgaoJulgador
   movimentos[].orgaoJulgador.codigo
   movimentos[].orgaoJulgador.nome
   nivelSigilo
   numeroProcesso
   orgaoJulgador
   orgaoJul

,linha,n_chaves_topo,chaves_topo,tem_hits,tem_movimentos,tem_numeroProcesso,n_caminhos
0,1,14,"@timestamp, assuntos, classe, dataAjuizamento, dataHoraUltimaAtualizacao, formato, g...",False,True,True,39
1,2,14,"@timestamp, assuntos, classe, dataAjuizamento, dataHoraUltimaAtualizacao, formato, g...",False,True,True,39
2,3,14,"@timestamp, assuntos, classe, dataAjuizamento, dataHoraUltimaAtualizacao, formato, g...",False,True,True,39
3,4,14,"@timestamp, assuntos, classe, dataAjuizamento, dataHoraUltimaAtualizacao, formato, g...",False,True,True,39
4,5,14,"@timestamp, assuntos, classe, dataAjuizamento, dataHoraUltimaAtualizacao, formato, g...",False,True,True,39
5,6,14,"@timestamp, assuntos, classe, dataAjuizamento, dataHoraUltimaAtualizacao, formato, g...",False,True,True,39
6,7,14,"@timestamp, assuntos, classe, dataAjuizamento, dataHoraUltimaAtualizacao, formato, g...",False,True,True,39
7,8,14,"@timestamp, assuntos, classe, dataAjuizamento, dataHoraUltimaAtualizacao, formato, g...",False,True,True,39
8,9,14,"@timestamp, assuntos, classe, dataAjuizamento, dataHoraUltimaAtualizacao, formato, g...",False,True,True,39
9,10,14,"@timestamp, assuntos, classe, dataAjuizamento, dataHoraUltimaAtualizacao, formato, g...",False,True,True,39


## Cobertura no ficheiro completo

Percorremos *todas* as linhas (streaming) para não assumir que a amostra das
20 primeiras é representativa. A tabela abaixo é o contrato de dados do resto
do pipeline: se um campo esperado tiver cobertura < 100 %, o parser usa o
nome real encontrado e documenta a diferença.


In [5]:
cobertura, meta = diagnostico_cobertura(caminho_jsonl, n_amostra_linhas=N_AMOSTRA)
print("Formato detectado:", meta["formato"])
print("Linhas no .jsonl:", meta["n_linhas_jsonl"])
print("Processos extraídos:", meta["n_processos"])
print("Páginas Elasticsearch:", meta["n_paginas_elasticsearch"])
print("Linhas não reconhecidas:", meta["n_linhas_nao_reconhecidas"])

catalogo.nova_tabela(
    cobertura.round({"pct_presente": 2}),
    titulo="Presença dos campos esperados da API DataJud no ficheiro de trabalho",
    nota=(
        "Campos conforme a documentação pública da API DataJud (CNJ). "
        f"N = {meta['n_processos']} processos extraídos de "
        f"{meta['n_linhas_jsonl']} linhas JSONL. Formato detectado: "
        f"{meta['formato']}."
    ),
    nome_ficheiro="tabela_cobertura_campos_datajud",
    interpretacao=(
        "A extração TRF2 usada neste pipeline tem um processo por linha JSONL "
        "(não uma página hits.hits). Os campos essenciais do ICP — sobretudo "
        "movimentos, dataAjuizamento e identificadores — devem aparecer com "
        "cobertura próxima de 100 %; ausências aqui condicionam a exclusão listwise."
    ),
)

caminhos_df = (
    pd.DataFrame(
        {"caminho": list(meta["caminhos_chaves"].keys()),
         "n_processos": list(meta["caminhos_chaves"].values())}
    )
    .assign(pct=lambda d: 100 * d["n_processos"] / meta["n_processos"])
    .sort_values("n_processos", ascending=False)
    .reset_index(drop=True)
)
display(caminhos_df.head(40))


Formato detectado: processo_por_linha
Linhas no .jsonl: 6300
Processos extraídos: 6300
Páginas Elasticsearch: 0
Linhas não reconhecidas: 0


campo,n_presente,n_ausente,pct_presente
numeroProcesso,6300.00,0.00,100.00
classe,6300.00,0.00,100.00
assuntos,6300.00,0.00,100.00
movimentos,6300.00,0.00,100.00
grau,6300.00,0.00,100.00
tribunal,6300.00,0.00,100.00
orgaoJulgador,6300.00,0.00,100.00
formato,6300.00,0.00,100.00
dataAjuizamento,6300.00,0.00,100.00
dataHoraUltimaAtualizacao,6300.00,0.00,100.00


,caminho,n_processos,pct
0,movimentos[].complementosTabelados[],6300,100.000000
1,movimentos[].codigo,6300,100.000000
2,movimentos[].orgaoJulgador,6300,100.000000
3,assuntos[].codigo,6300,100.000000
4,sistema.codigo,6300,100.000000
5,tribunal,6300,100.000000
6,assuntos[],6300,100.000000
7,orgaoJulgador.nome,6300,100.000000
8,movimentos[].dataHora,6300,100.000000
9,dataHoraUltimaAtualizacao,6300,100.000000


## Carregamento tabular

O módulo `src/io_datajud.py` projecta cada processo num DataFrame, **preservando**
a lista `movimentos` (insumo do índice) e expondo classe/tribunal/grau/órgão/formato
como colunas de *controlo e coorte* — não como componentes do ICP (secção 3.4).
`dataAjuizamento` nesta extração é `YYYYMMDDHHMMSS`, não ISO-8601; o parser trata
as duas formas.


In [6]:
processos = carregar_processos(caminho_jsonl)
print(processos.shape)
display(processos.drop(columns=["movimentos", "assuntos"]).head())

destino = DIRS["processado"] / "processos.parquet"
processos.to_parquet(destino, index=False)
print("Gravado:", destino)


(6300, 21)


,id_datajud,numero_processo,tribunal,grau,classe_codigo,classe_nome,orgao_codigo,orgao_nome,formato_codigo,formato_nome,nivel_sigilo,data_ajuizamento_bruta,data_ajuizamento,data_ultima_atualizacao_bruta,data_ultima_atualizacao,n_assuntos,assunto_principal_codigo,assunto_principal_nome,n_movimentos_bruto
0,TRF2_G1_00953390720164025117,00953390720164025117,TRF2,G1,1116,Execução Fiscal,12869,1ª Vara de Execução Fiscal de São Gonçalo,1,Eletrônico,0,20160720170200,2016-07-20 17:02:00,2025-02-13T20:32:38.132000Z,2025-02-13 20:32:38.132000+00:00,1,6017,Dívida Ativa (Execução Fiscal),11
1,TRF2_G1_00997536320164025112,00997536320164025112,TRF2,G1,1116,Execução Fiscal,17763,12ª Vara de Execução Fiscal do Rio de Janeiro,1,Eletrônico,0,20160726144900,2016-07-26 14:49:00,2025-02-13T20:32:47.210000Z,2025-02-13 20:32:47.210000+00:00,1,6017,Dívida Ativa (Execução Fiscal),34
2,TRF2_G1_00969703420164025101,00969703420164025101,TRF2,G1,1116,Execução Fiscal,14309,9ª Vara de Execução Fiscal do Rio de janeiro,1,Eletrônico,0,20160727142100,2016-07-27 14:21:00,2025-02-13T20:32:53.268000Z,2025-02-13 20:32:53.268000+00:00,1,10395,Multas e demais Sanções,13
3,TRF2_G1_00956253320164025101,00956253320164025101,TRF2,G1,7,Procedimento Comum Cível,12267,6ª Vara Federal do Rio de Janeiro,1,Eletrônico,0,20160718155500,2016-07-18 15:55:00,2025-02-13T20:32:57.363000Z,2025-02-13 20:32:57.363000+00:00,1,10159,Atualização de Conta,11
4,TRF2_G1_01011601320164025110,01011601320164025110,TRF2,G1,1116,Execução Fiscal,12893,1ª Vara de Execução Fiscal de São João de Meriti,1,Eletrônico,0,20160729082500,2016-07-29 08:25:00,2025-02-13T20:33:08.704000Z,2025-02-13 20:33:08.704000+00:00,1,10395,Multas e demais Sanções,8


Gravado: /workspace/data/processado/processos.parquet


## Síntese

- Cada linha do `.jsonl` corresponde a **um processo** (formato confirmado), não a uma página da API.
- Campos esperados: ver Tabela de cobertura; o nome real `numeroProcesso` coincide com a documentação; `dataAjuizamento` vem compacto.
- O campo central `movimentos[].codigo` / `dataHora` / `nome` está presente e é o insumo do ICP.
- Artefacto: `data/processado/processos.parquet` — ponto de entrada do notebook 02.
